# Compare NeuralLaplace encoder train modes
Этот ноутбук сравнивает 3 варианта `ENCODER_TRAIN_DATA_MODE`:
- `all`
- `random_half`
- `time_half`


## 1) Imports


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from run_benchmark import load_dataset
from bandit_benchmark import (
    NeuralLaplaceThompsonViaBayesianLogRegPolicy,
    RandomPolicy,
    core_scenarios,
    default_five_scenarios,
    run_scenarios,
)
from prepare_datasets import stage1_make_splits, stage2_scale_features


## 2) Config


In [ ]:
DATA_PATH = ROOT / 'data' / 'events.tsv'
DATASET_NAME = DATA_PATH.stem
ARTIFACTS_DIR = ROOT / 'artifacts' / DATASET_NAME
DATASETS_DIR = ARTIFACTS_DIR / 'datasets'

PREPARED_TRAIN_PATH = DATASETS_DIR / 'train_variant_1_scaled.parquet'
PREPARED_TEST_PATH = DATASETS_DIR / 'test_variant_1_scaled.parquet'

USE_PREPARED_SPLITS = True
TRAIN_DAYS = 1
SEED = 42
FULL_SCENARIOS = False

# neural params
NEURAL_HIDDEN_DIMS = [64, 32]
NN_EPOCHS = 10
NN_BATCH_SIZE = 256


## 3) Load / prepare dataset


In [ ]:
if USE_PREPARED_SPLITS and PREPARED_TRAIN_PATH.exists() and PREPARED_TEST_PATH.exists():
    train_df = pl.read_parquet(PREPARED_TRAIN_PATH)
    test_df = pl.read_parquet(PREPARED_TEST_PATH)
    print('loaded prepared splits')
else:
    train_s1, test_s1 = stage1_make_splits(str(DATA_PATH), str(DATASETS_DIR), TRAIN_DAYS, SEED)
    train_final, test_final = stage2_scale_features(train_s1, test_s1, str(DATASETS_DIR))
    train_df = pl.read_parquet(train_final)
    test_df = pl.read_parquet(test_final)

print('train:', train_df.height, 'test(random only):', test_df.height)


## 4) Compare 3 encoder modes


In [ ]:
policy_factories = {
    'neural_all': lambda: NeuralLaplaceThompsonViaBayesianLogRegPolicy(
        seed=SEED,
        hidden_dims=NEURAL_HIDDEN_DIMS,
        encoder_train_data_mode='all',
        nn_epochs=NN_EPOCHS,
        nn_batch_size=NN_BATCH_SIZE,
    ),
    'neural_random_half': lambda: NeuralLaplaceThompsonViaBayesianLogRegPolicy(
        seed=SEED,
        hidden_dims=NEURAL_HIDDEN_DIMS,
        encoder_train_data_mode='random_half',
        nn_epochs=NN_EPOCHS,
        nn_batch_size=NN_BATCH_SIZE,
    ),
    'neural_time_half': lambda: NeuralLaplaceThompsonViaBayesianLogRegPolicy(
        seed=SEED,
        hidden_dims=NEURAL_HIDDEN_DIMS,
        encoder_train_data_mode='time_half',
        nn_epochs=NN_EPOCHS,
        nn_batch_size=NN_BATCH_SIZE,
    ),
    # optional reference baseline
    'random': lambda: RandomPolicy(seed=SEED),
}

scenarios = default_five_scenarios() if FULL_SCENARIOS else core_scenarios()

result = run_scenarios(
    train_df=train_df,
    test_df=test_df,
    policy_factories=policy_factories,
    scenarios=scenarios,
    env_reward=None,
    show_progress=True,
)

metrics_df = result['metrics']
history_df = result['history']
trained_models = result['trained_models']

display(metrics_df.sort_values(['scenario', 'ips_ctr'], ascending=[True, False]).reset_index(drop=True))


## 5) Focus table: only neural variants


In [ ]:
neural_metrics = metrics_df[metrics_df['algo'].str.startswith('neural_')].copy()
display(neural_metrics.sort_values(['scenario', 'ips_ctr'], ascending=[True, False]).reset_index(drop=True))


## 6) Plot IPS CTR by mode


In [ ]:
if not neural_metrics.empty:
    for scenario_name, part in neural_metrics.groupby('scenario'):
        fig, ax = plt.subplots(figsize=(8, 4))
        part = part.sort_values('ips_ctr', ascending=False)
        ax.bar(part['algo'], part['ips_ctr'])
        ax.set_title(f'{scenario_name}: IPS CTR by encoder mode')
        ax.set_xlabel('model')
        ax.set_ylabel('ips_ctr')
        ax.tick_params(axis='x', rotation=25)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()
else:
    print('No neural metrics to plot.')


## 7) Embedding quality analysis (2D projection by action/reward)
Для каждого action строится 2D проекция выходов энкодера; цвет: `reward=0` vs `reward=1`.


In [ ]:
encoder_modes = ['neural_all', 'neural_random_half', 'neural_time_half']
MAX_ROWS = 5000
MAX_ACTIONS_PLOT = None  # например 10, чтобы ограничить число графиков

sample_pdf = test_df.to_pandas().copy()
if len(sample_pdf) > MAX_ROWS:
    sample_pdf = sample_pdf.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)

X = sample_pdf['features_list'].tolist()
actions = sample_pdf['show'].astype(int).to_numpy()
rewards = (sample_pdf['reward'].astype(float) > 0).astype(int).to_numpy()

for mode in encoder_modes:
    # берем первую доступную сценарию/модель
    policy_obj = None
    for scenario_name, models in trained_models.items():
        if mode in models:
            policy_obj = models[mode]
            break

    if policy_obj is None or getattr(policy_obj, '_encoder', None) is None:
        print(f'{mode}: encoder not available')
        continue

    Z = policy_obj._encoder.transform(X)

    try:
        from sklearn.decomposition import PCA
        Z2 = PCA(n_components=2, random_state=SEED).fit_transform(Z)
    except Exception:
        Z2 = Z[:, :2] if Z.shape[1] >= 2 else __import__('numpy').c_[Z[:, 0], Z[:, 0] * 0]

    emb_df = pd.DataFrame({
        'z1': Z2[:, 0],
        'z2': Z2[:, 1],
        'action': actions,
        'reward01': rewards,
    })

    action_values = sorted(emb_df['action'].unique())
    if MAX_ACTIONS_PLOT is not None:
        action_values = action_values[:MAX_ACTIONS_PLOT]

    for a in action_values:
        part = emb_df[emb_df['action'] == a]
        if part.empty:
            continue

        fig, ax = plt.subplots(figsize=(5, 4))
        part0 = part[part['reward01'] == 0]
        part1 = part[part['reward01'] == 1]

        ax.scatter(part0['z1'], part0['z2'], s=10, alpha=0.5, label='reward=0')
        ax.scatter(part1['z1'], part1['z2'], s=10, alpha=0.7, label='reward=1')

        ax.set_title(f'{mode}: action={a}')
        ax.set_xlabel('embedding_2d_1')
        ax.set_ylabel('embedding_2d_2')
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        plt.show()



## 8) Save comparison artifacts


In [ ]:
OUT_DIR = ARTIFACTS_DIR / 'encoder_mode_compare'
OUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUT_DIR / 'metrics_compare.csv'
history_path = OUT_DIR / 'history_compare.csv'

metrics_df.to_csv(metrics_path, index=False)
history_df.to_csv(history_path, index=False)

print('saved:', metrics_path)
print('saved:', history_path)
